## SELECCIÓN DE ATRIBUTOS MEDIANTE WRAPPERS
-  Autor: Germán Homero Morán Figueroa
- Descripción: La selección de atributos con wrappers es importante porque optimiza el rendimiento de un modelo de aprendizaje automático al elegir las variables más relevantes, en la predicción.
- Salida: Dataframe con las caracteristicas mas relevantes (20 ,30) para realizar pruebas con diferentes modelos de machine Learning

Existen 3 metodos a saber para realizar reducción de caracteristicas
- Filtros
- Wrappers o Envolturas
- Metodos integrados


!pip install mlxtend

In [1]:
# Librerias y Dependencias
# =================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.model_selection import GridSearchCV

In [3]:
print(tf.__version__)

2.17.0


In [ ]:
# Cargue de los datos
# ============================
#df = pd.read_excel("../../Data/Gold/DatasetFinal.xlsx")
df = pd.read_excel("../../Data/Gold/DatasetFinalFP.csv")
df.head(5)

,ID_LOTE,TIPO_SIEMBRA,SEM_TRATADAS,MATERIAL_GENETICO,CULT_ANT,DRENAJE,METODO_COSECHA,ALMACENAMIENTO_FINCA,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,...,Temp_Max_Avg_Mad,Temp_Min_Avg_Mad,Temp_Avg_Mad,Diurnal_Range_Avg_Mad,Sol_Ener_Accu_Mad,Temp_Max_34_Freq_Mad,Rain_Accu_Mad,Rain_10_Freq_Mad,Rhum_Avg_Mad,RDT_AJUSTADO
0,40,Mecanizado,NO,PIONEER 30F32,Algodon,SI,Manual,NO,5,63,...,32.05,23.60,27.83,8.45,13197.57,0.05,279.3,0.23,82.41,4767.44
1,43,Mecanizado,SI,DK 234,Maiz,SI,Manual,NO,5,64,...,32.37,23.49,27.93,8.89,12436.49,0.03,221.2,0.26,81.86,4651.16
2,44,Mecanizado,NO,PIONEER 30F32,Algodon,SI,Manual,NO,5,59,...,32.17,23.53,27.85,8.63,11267.17,0.03,226.0,0.27,82.61,5180.23
3,45,Mecanizado,NO,Otro,Algodon,SI,Manual,NO,5,64,...,32.19,23.54,27.86,8.65,11066.68,0.03,223.2,0.29,81.84,4897.67
4,46,Mecanizado,NO,Otro,Algodon,SI,Manual,NO,5,63,...,32.19,23.54,27.86,8.65,11066.68,0.03,223.2,0.29,81.84,5302.33


In [11]:
# Selección de las variables categoricas
# ======================================================
cat_features = df.select_dtypes(include = ["object", "category"]).columns
cat_features

Index(['TIPO_SIEMBRA', 'SEM_TRATADAS', 'MATERIAL_GENETICO', 'CULT_ANT',
       'DRENAJE', 'METODO_COSECHA', 'ALMACENAMIENTO_FINCA',
       'TERRENO_CIRCUN_RASTA', 'POSICION_PERFIL_RASTA', 'PEDREG_PERFIL_ROCAS',
       'CAP_ENDURE_RASTA', 'MOTEADOS_RASTA', 'MOTEADOS_MAS70cm._RASTA',
       'ESTRUCTURA_RASTA', 'OBSERVA_EROSION_RASTA', 'OBSERVA_MOHO_RASTA',
       'OBSERVA_COSTRAS_DURAS_RASTA', 'SITIO_EXPUESTO_SOL_RASTA',
       'OBSERVA_COSTRAS_BLANCAS_RASTA', 'OBSERVA_COSTAS_NEGRAS_RASTA',
       'REGION_SECA_ARIDA_RASTA', 'OBSERVA_RAICES_VIVAS_RASTA',
       'OBSERVA_PLANTAS_PEQUENAS_RASTA', 'OBSERVA_HOJARASCA_MO_RASTA',
       'SUELO_NEGRO_BLANDO_RASTA', 'CUCHILLO_PRIMER_HTE_RASTA',
       'CERCA_RIOS_QUEBRADAS_RASTA', 'RECUBRIMIENTO_VEGETAL__SUELO_RASTA',
       'd.interno', 'drenaje_externo'],
      dtype='object')

In [12]:
#  
# 1. Codificaicoón variables categoricas a formato one-hot
# 2. Eliminar variables originales 
# 3. Concatenar dataframe final con el dumificado 
# ========================================================
df_cat = pd.get_dummies(df[cat_features], drop_first = True, dummy_na = True)
df.drop(cat_features, axis = 1, inplace = True)
df_final = pd.concat([df,df_cat], axis = 1)

# División Features | Variable objetivo
# ===========================================================
Y = df_final.RDT_AJUSTADO
X = df_final.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1)


## Wrappers - Selección de Caracteristicas.

In [ ]:
# Seleccion de atributos mediante Wrappers
# ============================================================
excel = "../../Data/Silver/Otros/resultados_sa.xlsx"
k_features = [10 ,20]
contador_features = 0
iter_number = 3


for i in range(len(k_features)):
    for iter in range(iter_number):
        nomenclaturaHoja = "k_features_"+str(i)+"_"+str(iter)

        sfs = SFS(LinearRegression(),
                k_features=k_features[i],
                forward=True,
                floating=False,
                scoring = 'r2',
                cv = 10)

        # Ajuste del modelo
        sfs.fit(X, Y)
        # Obtengo los nombres de las caracteristicas | features mas relevantes
        tw_best_atri = sfs.k_feature_names_ 
        # Guardo los nombres de c/d iteracion en un excel para posteriores analisis
        df_features = pd.DataFrame(tw_best_atri).rename(columns={0: "Features"})

        # Apertura del excel
        with pd.ExcelWriter(excel,engine="openpyxl", mode = 'a', if_sheet_exists="overlay"
                                        ) as writer:
                        df_features.to_excel(writer, index=None, sheet_name=nomenclaturaHoja)

        contador_features= contador_features + 1
        print("Ejecucion_Numero_: ", contador_features)
    

Ejecucion_Numero_:  1
Ejecucion_Numero_:  2
Ejecucion_Numero_:  3
Ejecucion_Numero_:  4
Ejecucion_Numero_:  5
Ejecucion_Numero_:  6


In [ ]:
# Selección de Atributos
# ===================================================
sfs = SFS(LinearRegression(),
          k_features=10,
          forward=True,
          floating=False,
          scoring = 'r2',
          cv = 10)

sfs.fit(X, Y)
sfs.est_    

ElasticNet()

In [ ]:
# Score
# ===============================================
sfs.k_score_

-0.09820488474352124

In [23]:
# Caracteristicas seleccionadas durante el ajuste
# ===============================================
sfs.k_feature_names_ 

('TotN_Siem_Emer',
 'FerQui_Emer_Flor',
 'PROFUND_RAICES_VIVAS_RASTA',
 'Porc_A',
 'Porc_DURO',
 'Rain_Accu_Veg',
 'Temp_Min_Avg_Mad',
 'Temp_Avg_Mad',
 'METODO_COSECHA_Mecanizada',
 'OBSERVA_PLANTAS_PEQUENAS_RASTA_PLANTAS NORMALES')

## REFERENCIAS
-  https://www.analyticsvidhya.com/blog/2020/10/a-comprehensive-guide-to-feature-selection-using-wrapper-methods-in-python/